<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/homicidios/Homicidios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homicídios — taxa por 100 mil habitantes

Este notebook é o motor de coleta e publicação das séries `homicidios-anual`,
`homicidios-armas-fogo-anual` e `proporcao-homicidios-armas-fogo-anual`.

- **Fonte:** MS/SVS/CGIAE — Sistema de Informações sobre Mortalidade (SIM), compilado pelo
  IPEA/Atlas da Violência. Denominador: população residente — Estimativa/TCU (ajustada),
  série IPEADATA `AVIOL12_VIOLPOP`.
- **Endpoints:**
  - homicídios registrados (Brasil): `https://www.ipea.gov.br/dados-api/series-values/328/1`
  - homicídios por armas de fogo (Brasil, 1995–2013): `https://www.ipea.gov.br/dados-api/series-values/31/1`
  - população residente: `https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='AVIOL12_VIOLPOP')`
  - complemento 2014–2024: `https://forumseguranca.org.br/wp-content/uploads/2026/05/atlas-violencia-2026-relatorio-completo.pdf`
    (Atlas da Violência 2026, Tabela 11.1)
- **Definição:** taxa anual de homicídios por 100 mil habitantes, por local de residência.
  Total: CID-9 E960–E978 + E990–E999 (1995) e CID-10 X85–Y09 + Y35 (1996 em diante).
  Armas de fogo: CID-9 E965 (1995) e CID-10 X93–X95 + Y35.0 (1996 em diante).
  A proporção é a participação das armas de fogo no total de homicídios.
- **Fórmula:** `taxa = 100000 × homicídios ÷ população`;
  `proporção = 100 × homicídios_armas ÷ homicídios_total`.
  Cada ano usa a população do mesmo ano; não há denominador fixo nem taxa herdada.
- **Período publicado:** `1995-01-01` a `2024-01-01` nas três séries.
- **Arquivos gerados:** `indicadores/homicidios/homicidios-anual.json`,
  `indicadores/homicidios/homicidios-armas-fogo-anual.json` e
  `indicadores/homicidios/proporcao-homicidios-armas-fogo-anual.json`, todos com `versao: 1`.

## Decisão metodológica

A taxa pronta do Atlas não tem denominador explícito, e o Atlas usa populações de vintages
diferentes ao longo do tempo: a metadata da série "Taxa de homicídios registrados" declara
Contagem Populacional em 1996, Estimativas TCU em 1997–2003 e 2016 e IBGE/Diretoria de
Pesquisas em 2004–2015. Isso produz degraus artificiais na curva.

Por isso este notebook **recalcula** as três séries a partir dos numeradores do SIM e de uma
única série de população (`AVIOL12_VIOLPOP`, a mesma usada pelo Ipea como ponderador das taxas
de segurança pública). A taxa oficial do Atlas é buscada apenas para conferência. A
`AVIOL12_VIOLPOP` tem os anos de 2022 e 2023 interpolados exponencialmente entre as estimativas
de 2021 e 2024, conforme a metadata da própria série; é a limitação conhecida do denominador.
Ver `metodologia.md`.

**Complemento 2014–2024.** A série 31 da API do Ipea cobre 1995–2013 na safra antiga; de 2014
em diante ela fica atrás da safra revisada (o último ponto, 2023, é 30.789, e não há 2024). O
Atlas da Violência 2026 (FBSP/Ipea) publica a safra revisada na Tabela 11.1 para 2014–2024,
com a mesma definição (CID-10 X93–X95 + Y35.0, Brasil, por residência). Como não há publicação
machine-readable dessa safra — a API não foi atualizada e o site do FBSP só oferece o PDF —, o
notebook transcreve as 11 contagens e as aplica como um bloco 2014–2024. O bloco é contíguo;
2022 = 33.580 coincide nas duas safras, então não há degrau em 2022. A emenda fica em
2013/2014: 2013 continua na série 31 (40.369) e 2014 entra pelo Atlas (43.397), uma variação
de +7,5% (a safra nova daria +6,2% se 2013 também fosse 40.870; o resíduo é a revisão de 2013
que não foi aplicada). O bloco reproduz a variação de −8,8% de 2023 para 2024; usar só 2024
contra o 2023 antigo mostraria −3,0%. Ver `metodologia.md`.

## Execução sem sobrescrever o publicado

A variável de ambiente `HOMICIDIOS_SAIDA` redireciona os três arquivos:

```
HOMICIDIOS_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook Homicidios.ipynb
```


## Por que calculamos a taxa em vez de republicar a do Atlas

O Atlas da Violência publica a taxa de homicídios pronta, mas a série dele **troca de
denominador ao longo do tempo**: a população vem da Contagem Populacional em 1996, das
Estimativas TCU em 1997–2003 e 2016, e da Diretoria de Pesquisas do IBGE em 2004–2015. O
efeito é que a curva ganha degraus que não são violência — são troca de base populacional.
Em 2007 e 2012, por exemplo, a taxa publicada implica uma população de 184,0 e 194,0 milhões,
quando a série de população do próprio período dava 189,5 e 199,2 milhões.

Duas consequências práticas:

1. **A taxa é derivada aqui.** Lemos as contagens de mortes (séries 328 e 31) e dividimos por
   **uma única** série de população residente (`AVIOL12_VIOLPOP`) em todos os anos. A fórmula
   fica declarada e auditável, e o leitor pode refazer a conta com a população publicada.
2. **A nossa taxa não é idêntica à do Atlas.** Diverge até 0,78 por 100 mil (2012: 28,63 contra
   29,41 publicados), porque o denominador é outro. Isso é escolha, não erro: preferimos uma
   série internamente consistente e reproduzível a reproduzir o patchwork de vintages.

Pela mesma razão a **proporção** de armas de fogo é calculada entre as **contagens**
(`armas ÷ total`), e não entre as duas taxas — razão de taxas só coincide com razão de
contagens quando as duas usam o mesmo denominador, o que não era o caso (em 2019 dava 67,76%
contra os 66,38% corretos).

**Limitação conhecida:** a `AVIOL12_VIOLPOP` declara que 2022 e 2023 são interpolação
exponencial entre 2021 e 2024. O erro é pequeno e limitado — a projeção 2024 do IBGE dá
210,9 milhões para 2022 contra 212,8 da VIOLPOP (0,9%). Trocar de fonte resolveria a
interpolação, mas truncaria a série em 2000 ou reintroduziria dois denominadores.


## Coleta dos numeradores

As duas séries de contagem são lidas da API do Atlas da Violência 3.0.0, no recorte Brasil
(`tipo_regiao = 1`, `regiao_id = 1076`). A leitura é sempre da série inteira: não há modo
incremental nem leitura do JSON já publicado.

A série 31 é a fonte de 1995–2013. Para 2014–2024, o notebook usa o complemento explícito do
Atlas 2026 (Tabela 11.1), porque a API ainda não publica a safra revisada; ver a seção
seguinte.

As séries `20` (taxa total) e `35` (taxa de armas) são baixadas junto, mas **não** entram na
saída: servem para conferir por que a taxa pronta do Atlas não é reproduzível por
`homicídios ÷ população` com uma população única. A série 35 é da safra antiga e para em
2022; a partir de 2014 a diferença contra ela mede também a revisão do numerador, não só a
troca de denominador.


## Complemento 2014–2024 (Atlas da Violência 2026)

A API do Ipea (série 31) cobre 1995–2013 na safra antiga e termina em 2023 com 30.789
homicídios por arma de fogo — um valor que já não é o da safra revisada e não tem 2024. O
Atlas da Violência 2026 publica, na Tabela 11.1 (p. 151), a safra revisada para o Brasil:

| ano | homicídios por arma de fogo |
|---:|---:|
| 2014 | 43.397 |
| 2015 | 42.656 |
| 2016 | 45.616 |
| 2017 | 49.085 |
| 2018 | 42.908 |
| 2019 | 32.091 |
| 2020 | 35.828 |
| 2021 | 35.070 |
| 2022 | 33.580 |
| 2023 | 32.749 |
| 2024 | 29.870 |

A edição de 2025 já trazia 2013–2023 (2023 = 32.749); a de 2026 acrescenta 2024 = 29.870.

A definição é a mesma da série 31: CID-10 X93–X95 + Y35.0, óbitos por residência, contagem
(não taxa). O denominador continua sendo a `AVIOL12_VIOLPOP`, e a taxa é recalculada por
`100000 × contagem ÷ população`; a proporção é `100 × armas ÷ total`. O bloco é contíguo e
aplicado junto: 2022 = 33.580 coincide nas duas safras, então não há degrau em 2022; a emenda
fica em 2013/2014. A variação 2013→2014 é +7,5% (40.369 → 43.397); se 2013 também fosse da
safra nova (40.870, Tabela 11.1 de 2025), seria +6,2% — a diferença é o resíduo de 2013, que
continua na API. A variação 2023→2024 do bloco é −8,8%; 2024 sozinho contra o 2023 antigo
daria −3,0%, um degrau falso.

**Por que PDF e não uma fonte machine-readable?** A API do Ipea não tem 2024 nem a safra
revisada; o site do FBSP publica o relatório apenas em PDF; os microdados do SIM/DATASUS não
estão disponíveis de forma tabulada para 2024 neste ambiente. A transcrição é explícita e
conferida contra a Tabela 11.1 e contra a Tabela 11.3 (participação de 70,1% em 2024), e a
condição de revisão está em `metodologia.md`: quando a API publicar a safra 2026, o
bloco é removido e o bloco 2014–2024 é conferido contra a API. O 2013 da safra revisada
(40.870, Tabela 11.1 de 2025) **não** é aplicado sozinho: criaria uma ilha de safra entre
1995–2012 e 2014–2024. A emenda 2013/2014 fica em +7,5% e o bloco 1995–2013 é trocado
inteiro quando a API publicar a safra revisada.


In [1]:
from pathlib import Path
import json
import math
import os
import time
from decimal import Decimal, ROUND_HALF_UP

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name
PASTA = (
    RAIZ / "indicadores" / "homicidios"
    if (RAIZ / "indicadores" / "homicidios").is_dir()
    else RAIZ
)
DESTINO = Path(os.environ.get("HOMICIDIOS_SAIDA", PASTA))

INICIO = 1995
BRASIL = 1076
URL_ATLAS = "https://www.ipea.gov.br/dados-api/series-values/{serie}/1"
URL_POPULACAO = (
    "https://www.ipeadata.gov.br/api/odata4/"
    "ValoresSerie(SERCODIGO='AVIOL12_VIOLPOP')"
)
URL_ATLAS_2026 = (
    "https://forumseguranca.org.br/wp-content/uploads/2026/05/"
    "atlas-violencia-2026-relatorio-completo.pdf"
)

# Safra antiga da série 31 (API do Ipea) e o bloco 2014-2024 do Atlas 2026,
# Tabela 11.1 (p. 151). Ver a seção "Limitações" de homicidios-armas-fogo-anual em metodologia.md.
SAFRA_API_ANTIGA_2023 = 30789
COMPLEMENTO_ATLAS_2026 = {
    2014: 43397,
    2015: 42656,
    2016: 45616,
    2017: 49085,
    2018: 42908,
    2019: 32091,
    2020: 35828,
    2021: 35070,
    2022: 33580,
    2023: 32749,
    2024: 29870,
}


def baixar_json(url, tentativas=4, timeout=300):
    """Baixa JSON com espera crescente; a API do Ipea oscila."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def serie_brasil(serie):
    """Devolve {ano: valor} do recorte Brasil de uma série do Atlas."""
    bruto = baixar_json(URL_ATLAS.format(serie=serie))
    leituras = {}
    for item in bruto:
        if item.get("tipo_regiao") == 1 and item.get("regiao_id") == BRASIL:
            ano = int(item["periodo"][:4])
            if ano in leituras:
                raise ValueError(f"ano repetido na série {serie}: {ano}")
            leituras[ano] = float(item["valor"])
    if not leituras:
        raise ValueError(f"a série {serie} não retornou o recorte Brasil")
    return leituras


homicidios = serie_brasil(328)
armas = serie_brasil(31)
oficial_total = serie_brasil(20)
oficial_armas = serie_brasil(35)

if armas.get(2023) == SAFRA_API_ANTIGA_2023:
    if 2024 in armas:
        raise ValueError(
            "a série 31 tem 2024, mas 2023 ainda é a safra antiga (30.789); "
            "reveja o complemento do Atlas 2026 antes de publicar"
        )
    print("API do Ipea na safra antiga (2023=30.789, sem 2024);")
    print("aplicando o bloco 2014-2024 do Atlas da Violência 2026 (Tabela 11.1).")
    armas.update(COMPLEMENTO_ATLAS_2026)
elif 2024 not in armas:
    raise ValueError(
        "a série 31 não tem 2024 e não está no estado antigo esperado; "
        "reveja o complemento do Atlas 2026"
    )

print(f"homicídios registrados:        {len(homicidios)} anos ({min(homicidios)}–{max(homicidios)})")
print(f"homicídios por armas de fogo:  {len(armas)} anos ({min(armas)}–{max(armas)})")
print(f"taxa oficial total (conferência):   {len(oficial_total)} anos")
print(f"taxa oficial de armas (conferência): {len(oficial_armas)} anos")
print(f"Saída: {relativo(DESTINO)}")


API do Ipea na safra antiga (2023=30.789, sem 2024);
aplicando o bloco 2014-2024 do Atlas da Violência 2026 (Tabela 11.1).
homicídios registrados:        36 anos (1989–2024)
homicídios por armas de fogo:  45 anos (1980–2024)
taxa oficial total (conferência):   35 anos
taxa oficial de armas (conferência): 43 anos
Saída: indicadores/homicidios


## Coleta do denominador

A população é a série `AVIOL12_VIOLPOP` do Ipeadata: "População residente —
Estimativa/TCU (ajustada)". A metadata declara que ela é o ponderador usado nas taxas de
segurança pública do Atlas, que os ajustes judiciais não são considerados e que **2022 e 2023
foram interpolados exponencialmente entre as estimativas de 2021 e 2024**.

O endpoint OData não aceita filtro por território; a série completa (~79 MB, todas as
unidades) é baixada e filtrada para `NIVNOME = "Brasil"` / `TERCODIGO = "0"`. Isso é feito uma
vez por execução, sem cache e sem leitura do JSON publicado. O bloco 2014–2024 usa a mesma
população ano a ano; a interpolação de 2022–2023 é a limitação conhecida do denominador, não
do numerador.


In [2]:
bruto_populacao = baixar_json(URL_POPULACAO, timeout=600)
populacao = {}
for item in bruto_populacao["value"]:
    if item.get("NIVNOME") == "Brasil" and item.get("TERCODIGO") == "0":
        ano = int(item["VALDATA"][:4])
        populacao[ano] = float(item["VALVALOR"])

if not populacao:
    raise ValueError("a série AVIOL12_VIOLPOP não retornou o Brasil")

print(f"população: {len(populacao)} anos ({min(populacao)}–{max(populacao)})")
for ano in (1995, 2000, 2003, 2007, 2012, 2019, 2021, 2022, 2023, 2024):
    print(f"  {ano}: {populacao[ano]:,.0f} habitantes")


população: 46 anos (1980–2025)
  1995: 155,822,296 habitantes
  2000: 169,799,170 habitantes
  2003: 180,619,076 habitantes
  2007: 189,462,739 habitantes
  2012: 199,239,473 habitantes
  2019: 210,131,314 habitantes
  2021: 213,317,639 habitantes
  2022: 212,820,386 habitantes
  2023: 212,572,795 habitantes
  2024: 212,577,978 habitantes


## Cálculo das três séries

Cada taxa é `100000 × homicídios ÷ população` no mesmo ano. A proporção é
`100 × homicídios_armas ÷ homicídios_total` nos anos em que as duas contagens existem — não é
a razão entre as duas taxas (essa razão só coincide quando as duas usam o mesmo denominador,
que é justamente o que a taxa pronta do Atlas não garante).


In [3]:
def arredondar(valor):
    """Arredonda para 2 casas com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


comuns_total = sorted(ano for ano in homicidios if ano >= INICIO and ano in populacao)
comuns_armas = sorted(ano for ano in armas if ano >= INICIO and ano in populacao)
comuns_proporcao = sorted(set(comuns_total) & set(comuns_armas))

for nome, anos in (
    ("total", comuns_total),
    ("armas", comuns_armas),
    ("proporção", comuns_proporcao),
):
    if anos != list(range(INICIO, anos[-1] + 1)):
        raise ValueError(f"lacuna na série de {nome}: {anos}")

taxa_total = {ano: 100000 * homicidios[ano] / populacao[ano] for ano in comuns_total}
taxa_armas = {ano: 100000 * armas[ano] / populacao[ano] for ano in comuns_armas}
proporcao = {ano: 100 * armas[ano] / homicidios[ano] for ano in comuns_proporcao}

print(f"{'ano':>4} {'homicídios':>10} {'armas':>7} {'população':>13} {'taxa':>6} {'armas':>7} {'prop.%':>7}")
for ano in comuns_total:
    a = f"{int(armas[ano]):>7,}" if ano in armas else f"{'—':>7}"
    ta = f"{taxa_armas[ano]:>7.2f}" if ano in taxa_armas else f"{'—':>7}"
    p = f"{proporcao[ano]:>7.2f}" if ano in proporcao else f"{'—':>7}"
    print(
        f"{ano:>4} {homicidios[ano]:>10,.0f} {a} {populacao[ano]:>13,.0f} "
        f"{taxa_total[ano]:>6.2f} {ta} {p}"
    )

print()
print(f"total:     {len(taxa_total)} pontos ({min(taxa_total)}–{max(taxa_total)})")
print(f"armas:     {len(taxa_armas)} pontos ({min(taxa_armas)}–{max(taxa_armas)})")
print(f"proporção: {len(proporcao)} pontos ({min(proporcao)}–{max(proporcao)})")

# Conferência: a taxa oficial do Atlas não é reproduzível por contagem ÷ população.
print()
print("Conferência contra a taxa pronta do Atlas (série 35, safra antiga; mesmo recorte Brasil):")
print(f"{'ano':>4} {'oficial':>9} {'recalculada':>12} {'dif.':>7} | {'of. armas':>10} {'rec. armas':>11} {'dif.':>7}")
for ano in (2000, 2003, 2007, 2010, 2012, 2017, 2019, 2020, 2022, 2023, 2024):
    ot = oficial_total.get(ano)
    rt = taxa_total.get(ano)
    oa = oficial_armas.get(ano)
    ra = taxa_armas.get(ano)
    dt = f"{rt - ot:+.2f}" if ot is not None and rt is not None else "—"
    da = f"{ra - oa:+.2f}" if oa is not None and ra is not None else "—"
    print(
        f"{ano:>4} {(f'{ot:.2f}' if ot is not None else '—'):>9} "
        f"{(f'{rt:.2f}' if rt is not None else '—'):>12} {dt:>7} | "
        f"{(f'{oa:.2f}' if oa is not None else '—'):>10} "
        f"{(f'{ra:.2f}' if ra is not None else '—'):>11} {da:>7}"
    )


 ano homicídios   armas     população   taxa   armas  prop.%
1995     37,152  22,306   155,822,296  23.84   14.32   60.04
1996     38,929  22,976   157,070,163  24.78   14.63   59.02
1997     40,531  24,445   159,636,297  25.39   15.31   60.31
1998     41,974  25,674   161,790,182  25.94   15.87   61.17
1999     42,947  26,902   163,947,436  26.20   16.41   62.64
2000     45,433  30,865   169,799,170  26.76   18.18   67.94
2001     48,032  33,401   172,383,140  27.86   19.38   69.54
2002     49,816  34,160   174,630,259  28.53   19.56   68.57
2003     51,534  36,115   180,619,076  28.53   20.00   70.08
2004     48,909  34,187   182,908,586  26.74   18.69   69.90
2005     48,136  33,419   185,150,821  26.00   18.05   69.43
2006     49,704  34,921   187,335,127  26.53   18.64   70.26
2007     48,219  34,147   189,462,739  25.45   18.02   70.82
2008     50,659  35,676   191,532,851  26.45   18.63   70.42
2009     52,043  36,624   193,544,469  26.89   18.92   70.37
2010     53,016  36,792 

## Escrita dos JSONs do contrato

Cada arquivo é gravado inteiro, com todas as chaves da raiz e sem `rate` nem campo de
apresentação. As datas seguem a convenção do contrato: **dia 01** do início do período de
referência, uma por ano. O campo `fonte` declara a mistura: a série total continua na API do
Ipea (série 328); armas e proporção usam a série 31 até 2013 e o bloco 2014–2024 do Atlas
2026, com o PDF como `urlFonte`.


In [4]:
def pontos(serie):
    return [
        {"data": f"{ano}-01-01", "valor": arredondar(valor)}
        for ano, valor in sorted(serie.items())
    ]


FONTE_API = "IPEA/Atlas da Violência (SIM/DATASUS)"
FONTE_TOTAL = FONTE_API + "; taxa por 100 mil calculada pelo site"
FONTE_ARMAS = (
    FONTE_API
    + "; 2014–2024: Atlas da Violência 2026 (FBSP/Ipea), Tabela 11.1 (PDF)"
    + "; taxa por 100 mil calculada pelo site"
)
FONTE_PROPORCAO = (
    FONTE_API
    + "; 2014–2024: Atlas da Violência 2026 (FBSP/Ipea), Tabela 11.1 (PDF)"
    + "; proporção calculada pelo site"
)
series = {
    "homicidios-anual": {
        "versao": 1,
        "id": "homicidios-anual",
        "unidade": {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2},
        "eixo": "por-100-mil",
        "medida": "nivel",
        "granularidade": "anual",
        "fonte": FONTE_TOTAL,
        "urlFonte": "https://www.ipea.gov.br/atlasviolencia/dados-series/328/",
        "dados": pontos(taxa_total),
    },
    "homicidios-armas-fogo-anual": {
        "versao": 1,
        "id": "homicidios-armas-fogo-anual",
        "unidade": {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2},
        "eixo": "por-100-mil",
        "medida": "nivel",
        "granularidade": "anual",
        "fonte": FONTE_ARMAS,
        "urlFonte": URL_ATLAS_2026,
        "dados": pontos(taxa_armas),
    },
    "proporcao-homicidios-armas-fogo-anual": {
        "versao": 1,
        "id": "proporcao-homicidios-armas-fogo-anual",
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
        "eixo": "percentual",
        "medida": "participacao",
        "granularidade": "anual",
        "fonte": FONTE_PROPORCAO,
        "urlFonte": URL_ATLAS_2026,
        "dados": pontos(proporcao),
    },
}

DESTINO.mkdir(parents=True, exist_ok=True)
for nome, serie in series.items():
    arquivo = DESTINO / f"{nome}.json"
    with arquivo.open("w", encoding="utf-8") as saida:
        json.dump(serie, saida, ensure_ascii=False, indent=2)
        saida.write("\n")
    print(f"{len(serie['dados'])} pontos escritos em {arquivo}")


30 pontos escritos em indicadores/homicidios/homicidios-anual.json
30 pontos escritos em indicadores/homicidios/homicidios-armas-fogo-anual.json
30 pontos escritos em indicadores/homicidios/proporcao-homicidios-armas-fogo-anual.json


## Validação inline

A célula final falha se o contrato, a ordem, a unicidade, a continuidade dos anos, o dia 01, a
finitude dos valores, a fórmula ou a gravação em disco forem violados. Também confere os
números citados em `metodologia.md` e a proporção contra as duas contagens.


In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
ESPERADO = {
    "homicidios-anual": (
        {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2},
        "por-100-mil",
        "nivel",
        "anual",
    ),
    "homicidios-armas-fogo-anual": (
        {"tipo": "numero", "simbolo": "por 100 mil", "casas": 2},
        "por-100-mil",
        "nivel",
        "anual",
    ),
    "proporcao-homicidios-armas-fogo-anual": (
        {"tipo": "percentual", "simbolo": "%", "casas": 2},
        "percentual",
        "participacao",
        "anual",
    ),
}

for nome, serie in series.items():
    assert set(serie) == CHAVES_RAIZ, (nome, sorted(set(serie) ^ CHAVES_RAIZ))
    assert serie["versao"] == 1, (nome, serie["versao"])
    assert serie["id"] == nome, (nome, serie["id"])
    unidade, eixo, medida, granularidade = ESPERADO[nome]
    assert serie["unidade"] == unidade, (nome, serie["unidade"])
    assert serie["eixo"] == eixo, (nome, serie["eixo"])
    assert serie["medida"] == medida, (nome, serie["medida"])
    assert serie["granularidade"] == granularidade, (nome, serie["granularidade"])
    assert isinstance(serie["fonte"], str) and serie["fonte"], (nome, serie["fonte"])
    assert serie["urlFonte"].startswith("https://"), (nome, serie["urlFonte"])

    dados = serie["dados"]
    assert dados, f"{nome}: série vazia"
    datas = [ponto["data"] for ponto in dados]
    anos = [int(data[:4]) for data in datas]

    assert datas[0] == "1995-01-01", (nome, datas[0])
    assert all(data.endswith("-01-01") for data in datas), f"{nome}: dia diferente de 01"
    assert all(a < b for a, b in zip(datas, datas[1:])), f"{nome}: datas fora de ordem"
    assert anos == list(range(anos[0], anos[-1] + 1)), f"{nome}: ano faltando ou repetido"

    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, (nome, sorted(ponto))
        assert not isinstance(ponto["valor"], bool), (nome, ponto)
        assert isinstance(ponto["valor"], (int, float)), (nome, ponto)
        assert math.isfinite(ponto["valor"]), (nome, ponto)

    # Confere a gravação em disco.
    with (DESTINO / f"{nome}.json").open(encoding="utf-8") as arquivo:
        assert json.load(arquivo) == serie, f"{nome}: arquivo gravado difere do objeto"

# Fórmula: cada taxa é contagem ÷ população do mesmo ano; a proporção é contagem ÷ contagem.
for ponto in series["homicidios-anual"]["dados"]:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(100000 * homicidios[ano] / populacao[ano]), ponto

for ponto in series["homicidios-armas-fogo-anual"]["dados"]:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(100000 * armas[ano] / populacao[ano]), ponto

for ponto in series["proporcao-homicidios-armas-fogo-anual"]["dados"]:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(100 * armas[ano] / homicidios[ano]), ponto

# Complemento: o bloco 2014-2024 é exatamente o do Atlas 2026.
assert {ano: armas[ano] for ano in range(2014, 2025)} == COMPLEMENTO_ATLAS_2026
# 2022 coincide nas duas safras; a emenda é 2013/2014 (+7,5%) e 2023/2024 é -8,8%.
assert armas[2022] == 33580
assert round(100 * (armas[2014] / armas[2013] - 1), 1) == 7.5
assert round(100 * (armas[2024] / armas[2023] - 1), 1) == -8.8

# Cobertura: as três séries vão até 2024.
assert series["homicidios-anual"]["dados"][-1]["data"] == "2024-01-01"
assert series["homicidios-armas-fogo-anual"]["dados"][-1]["data"] == "2024-01-01"
assert series["proporcao-homicidios-armas-fogo-anual"]["dados"][-1]["data"] == "2024-01-01"


def valor(nome, ano):
    return {int(p["data"][:4]): p["valor"] for p in series[nome]["dados"]}[ano]


# Números citados na metodologia e na evidência.
assert valor("homicidios-anual", 2003) == 28.53
assert valor("homicidios-anual", 2007) == 25.45
assert valor("homicidios-anual", 2017) == 31.72
assert valor("homicidios-anual", 2019) == 21.65
assert valor("homicidios-anual", 2020) == 23.55
assert valor("homicidios-anual", 2022) == 21.81
assert valor("homicidios-anual", 2023) == 21.52
assert valor("homicidios-anual", 2024) == 20.04

assert valor("homicidios-armas-fogo-anual", 2003) == 20.00
assert valor("homicidios-armas-fogo-anual", 2007) == 18.02
assert valor("homicidios-armas-fogo-anual", 2014) == 21.40
assert valor("homicidios-armas-fogo-anual", 2015) == 20.86
assert valor("homicidios-armas-fogo-anual", 2016) == 22.13
assert valor("homicidios-armas-fogo-anual", 2017) == 23.74
assert valor("homicidios-armas-fogo-anual", 2018) == 20.58
assert valor("homicidios-armas-fogo-anual", 2019) == 15.27
assert valor("homicidios-armas-fogo-anual", 2020) == 16.92
assert valor("homicidios-armas-fogo-anual", 2021) == 16.44
assert valor("homicidios-armas-fogo-anual", 2022) == 15.78
assert valor("homicidios-armas-fogo-anual", 2023) == 15.41
assert valor("homicidios-armas-fogo-anual", 2024) == 14.05

assert valor("proporcao-homicidios-armas-fogo-anual", 2003) == 70.08
assert valor("proporcao-homicidios-armas-fogo-anual", 2007) == 70.82
assert valor("proporcao-homicidios-armas-fogo-anual", 2014) == 71.76
assert valor("proporcao-homicidios-armas-fogo-anual", 2015) == 72.20
assert valor("proporcao-homicidios-armas-fogo-anual", 2016) == 72.97
assert valor("proporcao-homicidios-armas-fogo-anual", 2017) == 74.82
assert valor("proporcao-homicidios-armas-fogo-anual", 2018) == 74.04
assert valor("proporcao-homicidios-armas-fogo-anual", 2019) == 70.53
assert valor("proporcao-homicidios-armas-fogo-anual", 2020) == 71.85
assert valor("proporcao-homicidios-armas-fogo-anual", 2021) == 73.30
assert valor("proporcao-homicidios-armas-fogo-anual", 2022) == 72.36
assert valor("proporcao-homicidios-armas-fogo-anual", 2023) == 71.59
assert valor("proporcao-homicidios-armas-fogo-anual", 2024) == 70.13

print(
    "ok: "
    f"{len(series['homicidios-anual']['dados'])} pontos no total (1995-01-01 a 2024-01-01), "
    f"{len(series['homicidios-armas-fogo-anual']['dados'])} em armas (1995-01-01 a 2024-01-01), "
    f"{len(series['proporcao-homicidios-armas-fogo-anual']['dados'])} na proporção "
    "(1995-01-01 a 2024-01-01); fórmula, dia 01 e contrato conferidos."
)


ok: 30 pontos no total (1995-01-01 a 2024-01-01), 30 em armas (1995-01-01 a 2024-01-01), 30 na proporção (1995-01-01 a 2024-01-01); fórmula, dia 01 e contrato conferidos.
